# Joining Land Registry Price Data with Energy Performance Certificates (2015 - 2025)

#### Load Price data

In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)


folder = Path('../data/raw')
file_gen = folder.glob('ppd_data*.csv')
pp_raw = pd.concat([pd.read_csv(f) for f in file_gen], ignore_index=True)
pp_raw['deed_date'] = pp_raw['deed_date'].astype('datetime64[ns]')

#### Load EPC data

In [2]:
epc_raw = pd.read_csv('../data/raw/EDC_OXFORDSHIRE_2015_2025.csv', usecols= [
    'certificate_number', 'address1', 'address2', 'address3', 'postcode',
    'posttown', 'address', 'local_authority', 'local_authority_label',
    'built_form', 'construction_age_band', 'current_energy_efficiency',
    'current_energy_rating', 'energy_consumption_current', 'extension_count', 'floor_height', 'heating_cost_current',
    'inspection_date', 'lodgement_date', 'number_habitable_rooms',
    'number_heated_rooms', 'potential_energy_efficiency', 'property_type',
    'tenure', 'total_floor_area', 'transaction_type', 'uprn', 'uprn_source'
])
epc_raw['inspection_date'] = epc_raw['inspection_date'].astype('datetime64[ns]')
epc_raw['lodgement_date'] = epc_raw['lodgement_date'].astype('datetime64[ns]')

#### Cleaning

Fix trailing whitespace

In [3]:
pp = pp_raw.copy()

for c in pp.select_dtypes(include=['string']).columns:
    pp[c] = pp[c].str.strip()
    print(f"{c} stripped")

unique_id stripped
postcode stripped
property_type stripped
new_build stripped
estate_type stripped
saon stripped
paon stripped
street stripped
locality stripped
town stripped
district stripped
county stripped
transaction_category stripped
linked_data_uri stripped


In [4]:
epc = epc_raw.copy()

for c in epc.select_dtypes(include=['string']).columns:
    epc[c] = epc[c].str.strip()
    print(f"{c} stripped")

certificate_number stripped
address1 stripped
address2 stripped
address3 stripped
postcode stripped
posttown stripped
address stripped
local_authority stripped
local_authority_label stripped
built_form stripped
construction_age_band stripped
current_energy_rating stripped
property_type stripped
tenure stripped
transaction_type stripped
uprn_source stripped


Remove commas

In [5]:
epc_address_cols = ['address1', 'address2', 'address3', 'postcode']
pp_address_cols = ['paon', 'saon', 'street', 'locality', 'postcode']

print(f"Commas replaced in epc:\n{epc[epc_address_cols].apply(lambda col: col.str.count(',')).sum()}")
print(f"Commas replaced in pp:\n{pp[pp_address_cols].apply(lambda col: col.str.count(',')).sum()}")

for c in epc_address_cols:
    epc[c] = epc[c].str.replace(',', '', regex=False)
for c in pp_address_cols:
    pp[c] = pp[c].str.replace(',', '', regex=False)

Commas replaced in epc:
address1    83515.0
address2     3910.0
address3      137.0
postcode        0.0
dtype: float64
Commas replaced in pp:
paon        4356.0
saon          33.0
street         0.0
locality       0.0
postcode       0.0
dtype: float64


Generate address token sets

In [6]:
pp['address'] = pp['paon'].str.cat([pp['saon'], pp['street'], pp['locality'], pp['postcode']], sep=' ', na_rep='')
print(pp['address'])
epc['address'] = epc['address1'].str.cat([epc['address2'], epc['address3'], epc['postcode']], sep=' ', na_rep='').apply(str.upper)
print(epc['address'])

0          CHILLING PLACE FARM PARK COTTAGE  BRILL HP18 9UH
1                        OAKCROFT FARM   BOARSTALL HP18 9UY
2                            TUTHER CORNER   BRILL HP18 9UZ
3                            TUTHER CORNER   BRILL HP18 9UZ
4         PIDDINGTON GATE FARM HEADLEY HOUSE  BRILL HP18...
                                ...                        
135552                   THE BUTCHERS GROUND   NORTH LEIGH 
135553                  TURLEY FARM LOTS 2 3 AND 4  HAILEY 
135554                        TURLEY LANE PLOT 154  HAILEY 
135555                       PLOTS 79-84   LONG HANBOROUGH 
135556                                114 GARAGE NEW ROAD  
Name: address, Length: 135557, dtype: str
0                                  10 NEWMAN ROAD   OX4 3UJ
1                        10 BARLOW CLOSE WHEATLEY  OX33 1NL
2                                65 CAMPBELL ROAD   OX4 3PG
3         12 ST. MICHAELS CLOSE SHIPTON-UNDER-WYCHWOOD  ...
4            FLAT 10 CANALSIDE HOUSE TRAMWAY ROAD  OX16 5B

In [7]:
pp['address_tokens'] = pp['address'].str.split().apply(frozenset)
epc['address_tokens'] = epc['address'].str.split().apply(frozenset)
print(epc['address_tokens'])

0                   frozenset({10, ROAD, NEWMAN, 3UJ, OX4})
1         frozenset({OX33, WHEATLEY, BARLOW, CLOSE, 1NL,...
2                 frozenset({3PG, ROAD, 65, CAMPBELL, OX4})
3         frozenset({6BE, SHIPTON-UNDER-WYCHWOOD, 12, CL...
4         frozenset({FLAT, OX16, CANALSIDE, ROAD, 5BG, H...
                                ...                        
216049                frozenset({30, OX12, APPLETONS, 7GG})
216050        frozenset({6ED, OX2, ROAD, WELL, WALTON, 1D})
216051             frozenset({8, OX2, ROAD, SYCAMORE, 9EJ})
216052         frozenset({4SW, CRESCENT, SPENCER, 68, OX4})
216053    frozenset({OX33, WHEATLEY, 1FQ, OCHRE, CLOSE, ...
Name: address_tokens, Length: 216054, dtype: object


#### Third Join (subset matching)

In [8]:
postcode_match = pp.merge(epc, on='postcode', how='inner')

def compare(row):
    return row['address_tokens_x'].issubset(row['address_tokens_y'])

postcode_match['result'] = postcode_match.apply(compare, axis=1)

In [9]:
epc['uprn'] = epc['uprn'].astype('Int64')

In [10]:
subset_matches = postcode_match[postcode_match['result']]

print(f"Third join (subset method) gives: {subset_matches['unique_id'].nunique()} matches. A matching rate of {100 * subset_matches['unique_id'].nunique()/pp.shape[0]:.1f}%")

Third join (subset method) gives: 105419 matches. A matching rate of 77.8%


Finding records where duplicate words were lost from the address in token setting

In [11]:
multi_match = subset_matches.groupby('unique_id')['uprn'].nunique()
multi_uprn = multi_match[multi_match > 1]

pp['token_count'] = pp['address'].str.count(r"\S+")
lost_tokens = pp[pp['token_count'] > pp['address_tokens'].apply(len)]
result = lost_tokens[lost_tokens['unique_id'].isin(multi_uprn.index)]

Date filtering

In [12]:
third_join = subset_matches[~subset_matches['unique_id'].isin(multi_uprn.index)]
third_join['time_to_sale'] = third_join['deed_date'] - third_join['inspection_date']

drop_neg = third_join[third_join['time_to_sale'] >= '0 days']
third_join_presale_epc = drop_neg.sort_values(by='time_to_sale', ascending=True).drop_duplicates(subset='unique_id')
print(f"Properties remaining with matched pre-sale EPCs: {third_join_presale_epc.shape[0]}")
print(f"Corresponds to a {100 * third_join_presale_epc.shape[0]/pp.shape[0]:.1f}% match rate")

Properties remaining with matched pre-sale EPCs: 91150
Corresponds to a 67.2% match rate


Drop matching and helping columns

In [13]:
cols_to_keep = ['unique_id', 'price_paid', 'deed_date', 'address_x', 'saon', 'paon', 'street', 'locality', 'town', 'district',  
                'postcode', 'property_type_x', 'new_build', 'estate_type',  'transaction_category',  'certificate_number', 
                'built_form', 'construction_age_band', 'current_energy_efficiency', 'current_energy_rating', 
                'energy_consumption_current', 'extension_count', 'floor_height', 'heating_cost_current', 
                'inspection_date', 'lodgement_date', 'number_habitable_rooms', 'number_heated_rooms', 
                'potential_energy_efficiency', 'property_type_y', 'tenure', 'total_floor_area', 
                'transaction_type', 'uprn', 'uprn_source', 'time_to_sale']

df = third_join_presale_epc[cols_to_keep]
df = df.rename(columns={'property_type_x':'property_subtype', 'property_type_y':'property_type', 'time_to_sale':'inspection_to_sale'})

df.dtypes

unique_id                                  str
price_paid                               int64
deed_date                       datetime64[ns]
address_x                                  str
saon                                       str
paon                                       str
street                                     str
locality                                   str
town                                       str
district                                   str
postcode                                   str
property_subtype                           str
new_build                                  str
estate_type                                str
transaction_category                       str
certificate_number                         str
built_form                                 str
construction_age_band                      str
current_energy_efficiency                int64
current_energy_rating                      str
energy_consumption_current               int64
extension_cou

Check date dtypes and ranges

In [14]:
dates = ['inspection_date', 'lodgement_date', 'deed_date']

print(df[dates].dtypes)

print(df[dates].agg(['min', 'max']).T)

inspection_date    datetime64[ns]
lodgement_date     datetime64[ns]
deed_date          datetime64[ns]
dtype: object
                       min        max
inspection_date 2013-10-31 2025-12-11
lodgement_date  2015-01-02 2025-12-13
deed_date       2015-01-07 2025-12-23


Create EPC age bands from 'inspection_to_sale'

In [15]:
df['inspection_to_sale'] = df['inspection_to_sale'].dt.days

bins = [0, 365, 1095, 1825, 3650, np.inf]
labels = ['0-1 years', '1-3 years', '3-5 years', '5-10 years', '10 years+']

df['EPC_age_band'] = pd.cut(df['inspection_to_sale'], bins=bins, labels=labels, include_lowest=True)
df['EPC_age_band'].value_counts().sort_index()

EPC_age_band
0-1 years     61097
1-3 years     14484
3-5 years      7388
5-10 years     8090
10 years+        91
Name: count, dtype: int64

Add flag for expired EPCs (10 years from lodgement_date)

In [16]:
df['EPC_expired'] = ((df['deed_date'] > (df['lodgement_date'] + pd.DateOffset(years=10))))
print(pd.concat([df['EPC_expired'].value_counts(),
df['EPC_expired'].value_counts(normalize=True)], axis=1))

             count  proportion
EPC_expired                   
False        91088     0.99932
True            62     0.00068


Adding datetime calendar features to joined/matched data

In [17]:
df['dayofweek'] = df['deed_date'].dt.dayofweek
df['day'] = df['deed_date'].dt.day
df['month'] = df['deed_date'].dt.month
df['quarter'] = df['deed_date'].dt.quarter
df['year'] = df['deed_date'].dt.year

print(df.sample())

                                   unique_id  price_paid  deed_date                address_x saon paon     street locality     town  district  postcode property_subtype new_build estate_type transaction_category        certificate_number     built_form         construction_age_band  current_energy_efficiency current_energy_rating  energy_consumption_current  extension_count  floor_height  heating_cost_current inspection_date lodgement_date  number_habitable_rooms  number_heated_rooms  potential_energy_efficiency property_type          tenure  total_floor_area transaction_type          uprn      uprn_source  inspection_to_sale EPC_age_band  EPC_expired  dayofweek  day  month  quarter  year
190955  0B853950-4D79-69A5-E063-4704A8C07DAC      230000 2023-08-31  97  WOODFIELD  OX16 1PY  NaN   97  WOODFIELD      NaN  BANBURY  CHERWELL  OX16 1PY                S         N           F                    A  2842-3025-6207-7077-3204  Semi-Detached  England and Wales: 1967-1975                    